# 01 — Scopus Data Collection & Aggregation Pipeline: *Big Earth Data* (2017–2026)

> **Note:** Pre-computed, anonymized, aggregated CSV datasets are **already included** in `data/`.  
> You **do not** need to run this notebook (or hold a Scopus API key) to reproduce all manuscript figures and tables — proceed directly to **`02_manuscript_figures_tables.ipynb`**.

This self-contained notebook documents and implements the complete upstream pipeline used to collect bibliographic and affiliation records from the **Elsevier Scopus API** and aggregate them into the anonymized summary CSVs in `data/`.

### Elsevier API Compliance & Reproducibility
* **No API keys included:** Copy `.env.example` to `.env` and supply your own institutional `SCOPUS_API_KEY_1` from <https://dev.elsevier.com/>.
* **No raw API payloads redistributed:** Raw Scopus JSON responses contain copyrighted abstracts and author profiles and are stored strictly in a git-ignored local directory (`raw/`), in compliance with Elsevier's Terms of Service.
* **Exact corpus snapshot (`data/corpus_index.csv`):** Because Scopus continuously indexes new 2026 publications after our **August 30, 2026** cutoff (`N = 356` indexed items; `339` citable), `USE_FROZEN_SNAPSHOT = True` fetches the exact 356 Scopus IDs recorded in `data/corpus_index.csv`. Set `USE_FROZEN_SNAPSHOT = False` to execute a live `SRCTITLE("Big Earth Data") AND (PUBYEAR > 2016)` Search API query instead.

## 1. Imports & Configuration

In [ ]:
import collections
import csv
import datetime
import json
import os
import re
import time
import urllib.parse
from itertools import combinations
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

# ── Load Scopus API key(s) from .env ─────────────────────────────────────────
load_dotenv(dotenv_path=Path('.env'))
API_KEYS = [os.environ.get(f'SCOPUS_API_KEY_{i}', '').strip() for i in range(1, 5)]
API_KEYS = [k for k in API_KEYS if k and k != 'your_scopus_api_key_here']
assert len(API_KEYS) > 0, (
    "No valid Scopus API key found. Copy `.env.example` to `.env` and set SCOPUS_API_KEY_1."
)
print(f"Loaded {len(API_KEYS)} Scopus API key(s).")

# ── Endpoints & study parameters ─────────────────────────────────────────────
API_SEARCH = 'https://api.elsevier.com/content/search/scopus'
API_ABSTRACT = 'https://api.elsevier.com/content/abstract/scopus_id/'

JOURNAL_TITLE = 'Big Earth Data'
START_YEAR = 2016  # PUBYEAR > 2016 captures 2017 onwards (journal launch)
USE_FROZEN_SNAPSHOT = True  # True = use exact Aug 30, 2026 corpus_index.csv (356 items)

SUBTYPES_CITABLE = {'Article', 'Review', 'Data Paper', 'Note'}

# ── Paths (all relative to release/) ─────────────────────────────────────────
DATA_DIR = Path('data')
RAW_DIR = Path('raw')
SEARCH_DIR = RAW_DIR / 'search'
RESPONSE_DIR = RAW_DIR / 'responses'
MANIFEST_PATH = SEARCH_DIR / 'manifest.json'
DERIVED_CSV = RAW_DIR / 'affil_BigEarthData.csv'
CORPUS_INDEX = DATA_DIR / 'corpus_index.csv'

for d in [DATA_DIR, SEARCH_DIR, RESPONSE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

## 2. HTTP Client & Rate-Limit Key Rotation

In [ ]:
class KeyRotator:
    """Rotate across configured Scopus API keys on HTTP 429/401/403."""
    def __init__(self, keys: list[str]):
        self.keys = keys
        self._idx = 0

    @property
    def current(self) -> str:
        return self.keys[self._idx]

    def rotate(self) -> None:
        self._idx += 1
        if self._idx >= len(self.keys):
            self._idx = 0
            print("All API keys cycled — sleeping 60s before retry...")
            time.sleep(60)
        else:
            print(f"Rotated to API key {self._idx + 1}/{len(self.keys)}")

    def headers(self) -> dict[str, str]:
        return {'X-ELS-APIKey': self.current, 'Accept': 'application/json'}


rotator = KeyRotator(API_KEYS)


def safe_get(url: str, max_retries: int = 3) -> requests.Response | None:
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, headers=rotator.headers(), timeout=30)
            if resp.status_code == 200:
                time.sleep(0.5)
                return resp
            if resp.status_code in (429, 401, 403):
                print(f"  HTTP {resp.status_code} — rotating key")
                rotator.rotate()
            else:
                wait = 2 ** attempt
                print(f"  HTTP {resp.status_code} — retry {attempt + 1} in {wait}s")
                time.sleep(wait)
        except requests.RequestException as exc:
            print(f"  Network error: {exc}")
            time.sleep(2 ** attempt)
    return None

## 3. Phase 1 — Corpus Discovery (Frozen Snapshot or Live Scopus Search API)

In [ ]:
SEARCH_FIELDS = {
    'scopus_id': 'dc:identifier',
    'title': 'dc:title',
    'creator': 'dc:creator',
    'journal': 'prism:publicationName',
    'year': 'prism:coverDisplayDate',
    'doi': 'prism:doi',
    'citedby_count': 'citedby-count',
    'subtype': 'subtypeDescription',
    'openaccess': 'openaccess',
}

if USE_FROZEN_SNAPSHOT and CORPUS_INDEX.exists():
    idx_df = pd.read_csv(CORPUS_INDEX, dtype=str)
    search_records = [
        {
            'scopus_id': f"SCOPUS_ID:{r['scopus_id']}",
            'year': str(r['year']),
            'doi': r.get('doi', '') or '',
            'subtype': r['subtype'],
            'openaccess': '1',  # Verified via Abstract Retrieval coredata below
        }
        for _, r in idx_df.iterrows()
    ]
    print(f"Loaded frozen August 30, 2026 snapshot from {CORPUS_INDEX}: {len(search_records)} records.")
else:
    query = f'SRCTITLE("{JOURNAL_TITLE}") AND (PUBYEAR > {START_YEAR})'
    run_ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    print(f"Running live Scopus Search query: {query}")

    search_records = []
    seen_ids = set()
    page = 0
    url = (
        f"{API_SEARCH}?query={urllib.parse.quote(query)}"
        f"&count=25&field={','.join(SEARCH_FIELDS.values())}"
    )

    while url:
        resp = safe_get(url)
        if resp is None:
            raise RuntimeError("Search API failed — check API key or institutional access.")

        raw_page = resp.json()
        (SEARCH_DIR / f"run_{run_ts}_page{page:03d}.json").write_text(
            json.dumps(raw_page, ensure_ascii=False), encoding='utf-8'
        )
        sr = raw_page.get('search-results', {})
        for entry in sr.get('entry', []):
            rec = {col: entry.get(sk, '') for col, sk in SEARCH_FIELDS.items()}
            if rec['scopus_id'] and rec['scopus_id'] not in seen_ids:
                search_records.append(rec)
                seen_ids.add(rec['scopus_id'])

        page += 1
        url = None
        for link in sr.get('link', []):
            if link.get('@ref') == 'next' and link.get('@_fa') == 'true':
                url = link.get('@href')
                break

    MANIFEST_PATH.write_text(
        json.dumps({'collected_at': run_ts, 'records': search_records}, indent=2),
        encoding='utf-8',
    )
    print(f"Live search complete: {len(search_records)} unique records across {page} pages.")

print("Subtype breakdown:", dict(collections.Counter(r['subtype'] for r in search_records)))

## 4. Phase 2 — Scopus Abstract Retrieval API (Affiliation Enrichment)

In [ ]:
already_cached = {p.stem for p in RESPONSE_DIR.glob('*.json')}
to_fetch = [
    r for r in search_records
    if r['scopus_id'].replace('SCOPUS_ID:', '') not in already_cached
]
print(f"Cached responses: {len(already_cached)} | Remaining to fetch: {len(to_fetch)}")

fetched, failed = 0, 0
for i, rec in enumerate(to_fetch, start=1):
    sid = rec['scopus_id'].replace('SCOPUS_ID:', '')
    resp = safe_get(API_ABSTRACT + sid)
    if resp is None:
        print(f"  FAILED: {sid}")
        failed += 1
        continue
    (RESPONSE_DIR / f"{sid}.json").write_text(
        json.dumps(resp.json(), ensure_ascii=False), encoding='utf-8'
    )
    fetched += 1
    if i % 25 == 0:
        print(f"  Progress: {i}/{len(to_fetch)} (fetched={fetched}, failed={failed})")

print(f"Abstract retrieval finished: fetched={fetched}, failed={failed}, total cached={len(list(RESPONSE_DIR.glob('*.json')))}")

## 5. Phase 3 — Parse Affiliations & Aggregate Anonymized Datasets

In [ ]:
COUNTRY_MAP = {
    'Viet Nam': 'Vietnam',
    'United States': 'United States of America',
    'Tanzania': 'United Republic of Tanzania',
    'Serbia': 'Republic of Serbia',
    'Russian Federation': 'Russia',
    'North Macedonia': 'Macedonia',
    'Macao': 'Macao S.A.R',
    'Hong Kong': 'Hong Kong S.A.R.',
    "Cote d'Ivoire": 'Ivory Coast',
    'Congo': 'Democratic Republic of the Congo',
    'Syrian Arab Republic': 'Syria',
    'Korea': 'South Korea',
    'Czech Republic': 'Czechia',
    'Iran': 'Iran',
    'Taiwan': 'Taiwan',
}


def extract_year(val: str) -> int | None:
    m = re.search(r'\b(20\d{2})\b', str(val))
    return int(m.group(1)) if m else None


def extract_affiliations(raw_json: dict) -> tuple[list[dict], bool]:
    """Extract institution affiliations and open-access flag from Abstract API payload."""
    inner = raw_json.get('abstracts-retrieval-response', {})
    core = inner.get('coredata', {}) or {}
    oa_val = core.get('openaccess', '1')
    is_oa = str(oa_val) == '1'

    raw_affil = inner.get('affiliation', [])
    if isinstance(raw_affil, dict):
        raw_affil = [raw_affil]
    if not isinstance(raw_affil, list):
        raw_affil = []

    affils = [
        {
            'affilname': (a.get('affilname') or '').strip(),
            'affiliation-country': (a.get('affiliation-country') or '').strip(),
            'affiliation-city': (a.get('affiliation-city') or '').strip(),
        }
        for a in raw_affil
    ]
    return affils, is_oa


articles = {}
subtype_year = collections.defaultdict(collections.Counter)
geography = {}
geo_year = collections.defaultdict(collections.Counter)
affiliation_yr = collections.defaultdict(collections.Counter)
collab_pairs = collections.Counter()

for rec in search_records:
    yr = extract_year(rec.get('year', ''))
    if yr is None:
        continue
    subtype = rec.get('subtype', '')
    sid = rec['scopus_id'].replace('SCOPUS_ID:', '')

    if yr not in articles:
        articles[yr] = {'total': 0, 'citable': 0, 'oa': 0, 'multi_country': 0}
    articles[yr]['total'] += 1
    subtype_year[yr][subtype] += 1

    if subtype not in SUBTYPES_CITABLE:
        continue

    json_path = RESPONSE_DIR / f"{sid}.json"
    if not json_path.exists():
        raise FileNotFoundError(f"Missing cached Abstract API response: {json_path}")

    raw_payload = json.loads(json_path.read_text(encoding='utf-8'))
    affil_list, is_oa = extract_affiliations(raw_payload)
    if str(rec.get('openaccess', '')) in ('0', '1'):
        is_oa = str(rec['openaccess']) == '1'

    articles[yr]['citable'] += 1
    if is_oa:
        articles[yr]['oa'] += 1

    countries_seen = []
    cities_seen = []
    for inst in affil_list:
        raw_c = inst['affiliation-country']
        country = COUNTRY_MAP.get(raw_c, raw_c) if raw_c else 'Unknown'
        city = inst['affiliation-city'] or 'Unknown'
        inst_name = inst['affilname']

        if country not in geography:
            geography[country] = {'num': 0, 'num_oa': 0, 'cities': collections.Counter()}

        # Full-score counting: each unique country counted once per citable item
        if country not in countries_seen:
            geography[country]['num'] += 1
            geo_year[yr][country] += 1
            if is_oa:
                geography[country]['num_oa'] += 1
            countries_seen.append(country)

        if (country, city) not in cities_seen:
            geography[country]['cities'][city] += 1
            cities_seen.append((country, city))

        if inst_name:
            affiliation_yr[yr][inst_name] += 1

    unique_countries = list(dict.fromkeys(c for c in countries_seen if c and c != 'Unknown'))
    if len(unique_countries) > 1:
        articles[yr]['multi_country'] += 1
        for pair in combinations(sorted(unique_countries), 2):
            collab_pairs[pair] += 1

# ── Write aggregated CSVs to data/ ───────────────────────────────────────────
all_subtypes = sorted({st for yc in subtype_year.values() for st in yc})
with open(DATA_DIR / 'SUBTYPE_YEAR.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f, lineterminator='\n')
    w.writerow(['year'] + all_subtypes)
    for yr in sorted(subtype_year):
        w.writerow([yr] + [subtype_year[yr].get(st, 0) for st in all_subtypes])

with open(DATA_DIR / 'metrics_summary.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f, lineterminator='\n')
    w.writerow(['year', 'total_items', 'citable', 'articles', 'reviews', 'notes',
                'non_citable', 'open_access', 'multi_country', 'oa_rate', 'collab_rate'])
    for yr in sorted(articles):
        a = articles[yr]
        sy = subtype_year[yr]
        cit_n = a['citable']
        w.writerow([
            yr, a['total'], cit_n,
            sy.get('Article', 0), sy.get('Review', 0),
            sy.get('Data Paper', 0) + sy.get('Note', 0),
            sy.get('Editorial', 0) + sy.get('Erratum', 0),
            a['oa'], a['multi_country'],
            round(a['oa'] / cit_n * 100, 1) if cit_n else 0.0,
            round(a['multi_country'] / cit_n * 100, 1) if cit_n else 0.0,
        ])

with open(DATA_DIR / 'COUNTRIES.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f, lineterminator='\n')
    w.writerow(['country', 'num', 'num_oa'])
    for c in sorted(geography, key=lambda x: (-geography[x]['num'], x)):
        w.writerow([c, geography[c]['num'], geography[c]['num_oa']])

with open(DATA_DIR / 'COUNTRIES_YEAR.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f, lineterminator='\n')
    w.writerow(['year', 'country', 'num'])
    for yr in sorted(geo_year):
        for c, cnt in sorted(geo_year[yr].items(), key=lambda x: (-x[1], x[0])):
            w.writerow([yr, c, cnt])

with open(DATA_DIR / 'AFFILIATIONS.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f, lineterminator='\n')
    w.writerow(['year', 'institution', 'num'])
    for yr in sorted(affiliation_yr):
        for inst, cnt in sorted(affiliation_yr[yr].items(), key=lambda x: (-x[1], x[0])):
            w.writerow([yr, inst, cnt])

with open(DATA_DIR / 'CITIES.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f, lineterminator='\n')
    w.writerow(['country', 'city', 'num'])
    for c in sorted(geography):
        for city, cnt in sorted(geography[c]['cities'].items(), key=lambda x: (-x[1], x[0])):
            w.writerow([c, city, cnt])

with open(DATA_DIR / 'COLLAB.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f, lineterminator='\n')
    w.writerow(['country_1', 'country_2', 'num'])
    for (c1, c2), cnt in sorted(collab_pairs.items(), key=lambda x: (-x[1], x[0])):
        w.writerow([c1, c2, cnt])

print(f"Aggregated CSVs written to {DATA_DIR}/ (citable items={sum(a['citable'] for a in articles.values())}, countries={len(geography)}).")

## 6. Phase 4 — Incremental City Geocoding (OpenStreetMap Nominatim)

In [ ]:
from geopy.extra.rate_limiter import RateLimiter
from geopy.geocoders import Nominatim

cities_df = pd.read_csv(DATA_DIR / 'CITIES.csv')
geocoded_path = DATA_DIR / 'CITIES_GEOCODED.csv'

if geocoded_path.exists():
    existing = pd.read_csv(geocoded_path)[['country', 'city', 'lat', 'lon']].drop_duplicates(['country', 'city'])
    merged = cities_df.merge(existing, on=['country', 'city'], how='left')
else:
    merged = cities_df.assign(lat=float('nan'), lon=float('nan'))

missing = merged[merged['lat'].isna() & (merged['city'].str.lower() != 'unknown')][['country', 'city']].drop_duplicates()
print(f"Cities already geocoded: {merged['lat'].notna().sum()} | Missing: {len(missing)}")

# Manual fallback coordinates for compound/institutional city strings in Scopus
MANUAL_CITY_COORDS = {
    ('Canada', 'Ottawa and Cambridge Bay'): (45.4215, -75.6972),
    ('Ethiopia', 'Makale'): (13.4967, 39.4753),
    ('Luxembourg', 'Luxembourg-City Incubator'): (49.6116, 6.1319),
}

if len(missing) > 0:
    geolocator = Nominatim(user_agent='big_earth_data_scientometrics_2026', timeout=10)
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.1)
    new_coords = {}
    for _, row in missing.iterrows():
        key = (row['country'], row['city'])
        if key in MANUAL_CITY_COORDS:
            new_coords[key] = MANUAL_CITY_COORDS[key]
            continue
        q = f"{row['city']}, {row['country']}"
        loc = geocode(q)
        if not loc and row['country'] == 'United States of America':
            loc = geocode(f"{row['city']}, USA")
        if loc:
            new_coords[key] = (round(loc.latitude, 4), round(loc.longitude, 4))
    for idx in merged.index:
        if pd.isna(merged.loc[idx, 'lat']):
            key = (merged.loc[idx, 'country'], merged.loc[idx, 'city'])
            if key in new_coords:
                merged.loc[idx, 'lat'], merged.loc[idx, 'lon'] = new_coords[key]

final_geo = merged.dropna(subset=['lat', 'lon'])
final_geo.to_csv(geocoded_path, index=False, lineterminator='\n')
print(f"Saved {len(final_geo)} geocoded city records to {geocoded_path}.")